In [2]:
from google.colab import drive
import pandas as pd
drive.mount('/content/drive')
BASE_PATH = '/content/drive/MyDrive/Cadetx Project/HeavySuppliersWarehouseDatasets/'

branches = pd.read_csv(BASE_PATH + 'branches.csv')
ledger   = pd.read_csv(BASE_PATH + 'stock_ledger.csv')
so_h     = pd.read_csv(BASE_PATH + 'sales_orders_header.csv')
so_l     = pd.read_csv(BASE_PATH + 'sales_orders_lines.csv')
po_h     = pd.read_csv(BASE_PATH + 'purchase_orders_header.csv')
po_l     = pd.read_csv(BASE_PATH + 'purchase_orders_lines.csv')

for name, df in [('branches', branches), ('ledger', ledger), ('so_h', so_h),
                 ('so_l', so_l), ('po_h', po_h), ('po_l', po_l)]:
    print(name, df.shape)
    print(list(df.columns))
    print()

Mounted at /content/drive
branches (6, 13)
['branch_id', 'branch_name', 'city', 'state', 'region', 'warehouse_type', 'warehouse_capacity', 'service_center_available', 'manager_id', 'total_employees', 'avg_monthly_revenue', 'monthly_operational_cost', 'market_demand_index']

ledger (237230, 9)
['movement_id', 'product_id', 'branch_id', 'movement_type', 'movement_date', 'quantity', 'reference_type', 'reference_id', 'running_balance']

so_h (20000, 11)
['so_id', 'customer_id', 'branch_id', 'order_date', 'delivery_date', 'order_status', 'payment_terms', 'total_order_value', 'total_gst_amount', 'grand_total', 'sales_channel']

so_l (130402, 9)
['so_id', 'line_number', 'product_id', 'quantity', 'unit_price', 'gst_rate', 'line_total', 'gst_amount', 'line_grand_total']

po_h (24000, 10)
['po_id', 'supplier_id', 'branch_id', 'order_date', 'expected_delivery_date', 'received_date', 'po_status', 'total_cost', 'total_gst_amount', 'grand_total']

po_l (155495, 9)
['po_id', 'line_number', 'product_i

In [3]:
print('so_l lines whose so_id is not in so_h:', (~so_l['so_id'].isin(so_h['so_id'])).sum())
print('po_l lines whose po_id is not in po_h:', (~po_l['po_id'].isin(po_h['po_id'])).sum())
print('so_h branch_id not in branches:', (~so_h['branch_id'].isin(branches['branch_id'])).sum())
print('po_h branch_id not in branches:', (~po_h['branch_id'].isin(branches['branch_id'])).sum())
print('branches in so_h:', sorted(so_h['branch_id'].unique()))
print('branches in po_h:', sorted(po_h['branch_id'].unique()))



so_l lines whose so_id is not in so_h: 0
po_l lines whose po_id is not in po_h: 0
so_h branch_id not in branches: 0
po_h branch_id not in branches: 0
branches in so_h: ['AHM001', 'CHN001', 'DEL001', 'HYD001', 'KOL001', 'PUN001']
branches in po_h: ['AHM001', 'CHN001', 'DEL001', 'HYD001', 'KOL001', 'PUN001']


In [4]:
sales = so_l.merge(so_h[so_h['order_status'] == 'Delivered'][['so_id', 'branch_id', 'order_date', 'delivery_date']], on='so_id', how='inner')
purch = po_l.merge(po_h[po_h['po_status'] == 'Received'][['po_id', 'branch_id', 'order_date', 'received_date']], on='po_id', how='inner')

print('sales (Delivered lines):', len(sales), '| units:', sales['quantity'].sum())
print('purch (Received lines):', len(purch), '| units:', purch['quantity'].sum())
print('sales missing branch or delivery_date:', sales[['branch_id', 'delivery_date']].isna().sum().sum())
print('purch missing branch or received_date:', purch[['branch_id', 'received_date']].isna().sum().sum())

sales (Delivered lines): 117656 | units: 1234939
purch (Received lines): 140077 | units: 22387123
sales missing branch or delivery_date: 0
purch missing branch or received_date: 0


In [5]:
out_t = sales.groupby('branch_id').agg(units_out=('quantity', 'sum'), lines_out=('quantity', 'size'), orders_out=('so_id', 'nunique'))
in_t = purch.groupby('branch_id').agg(units_in=('quantity', 'sum'), lines_in=('quantity', 'size'), orders_in=('po_id', 'nunique'))
t = in_t.join(out_t)
t['units_total'] = t['units_in'] + t['units_out']
t['share_pct'] = (100 * t['units_total'] / t['units_total'].sum()).round(2)
print(t)
print()
print('units_in total:', t['units_in'].sum(), '| units_out total:', t['units_out'].sum())
print('orders_in total:', t['orders_in'].sum(), '| orders_out total:', t['orders_out'].sum())

           units_in  lines_in  orders_in  units_out  lines_out  orders_out  \
branch_id                                                                    
AHM001      3681090     23104       3603     214223      20425        3115   
CHN001      3797868     23655       3633     253921      24202        3677   
DEL001      3777619     23607       3616     185319      17511        2682   
HYD001      3718428     23217       3606     168975      16102        2478   
KOL001      3661187     22919       3547     230868      21989        3413   
PUN001      3750931     23575       3625     181633      17427        2668   

           units_total  share_pct  
branch_id                          
AHM001         3895313      16.49  
CHN001         4051789      17.15  
DEL001         3962938      16.78  
HYD001         3887403      16.46  
KOL001         3892055      16.48  
PUN001         3932564      16.65  

units_in total: 22387123 | units_out total: 1234939
orders_in total: 21630 | orders_ou

In [6]:
sales['year'] = pd.to_datetime(sales['delivery_date']).dt.year
purch['year'] = pd.to_datetime(purch['received_date']).dt.year

out_y = sales.pivot_table(index='branch_id', columns='year', values='quantity', aggfunc='sum', margins=True, margins_name='TOTAL')
in_y = purch.pivot_table(index='branch_id', columns='year', values='quantity', aggfunc='sum', margins=True, margins_name='TOTAL')
print('UNITS OUT per year:')
print(out_y)
print()
print('UNITS IN per year:')
print(in_y)

UNITS OUT per year:
year         2019    2020    2021    2022    2023    2024  2025    TOTAL
branch_id                                                               
AHM001      33943   39000   36971   33767   38048   31921   573   214223
CHN001      46886   40613   40806   41049   43376   40818   373   253921
DEL001      28812   30422   30001   34287   29414   31967   416   185319
HYD001      27519   29400   28210   27715   27763   27816   552   168975
KOL001      37478   39862   41681   36657   37610   36941   639   230868
PUN001      29948   31893   29266   29241   29367   31099   819   181633
TOTAL      204586  211190  206935  202716  205578  200562  3372  1234939

UNITS IN per year:
year          2019     2020     2021     2022     2023     2024    2025  \
branch_id                                                                 
AHM001      581390   629042   658827   566431   599234   615796   30370   
CHN001      606670   676736   567223   635957   635778   642929   32575   
DEL

In [7]:
sales_m = sales[sales['year'] <= 2024].copy()
sales_m['month'] = pd.to_datetime(sales_m['delivery_date']).dt.to_period('M')
purch_m = purch[purch['year'] <= 2024].copy()
purch_m['month'] = pd.to_datetime(purch_m['received_date']).dt.to_period('M')

mo = sales_m.groupby(['month', 'branch_id'])['quantity'].sum().unstack('branch_id')
mi = purch_m.groupby(['month', 'branch_id'])['quantity'].sum().unstack('branch_id')

res = pd.DataFrame({
    'months_out': mo.count(), 'avg_out': mo.mean().round(0), 'min_out': mo.min(), 'max_out': mo.max(),
    'months_in': mi.count(), 'avg_in': mi.mean().round(0), 'min_in': mi.min(), 'max_in': mi.max()})
print(res)

           months_out  avg_out  min_out  max_out  months_in   avg_in  min_in  \
branch_id                                                                      
AHM001             72   2967.0     1294     4150         72  50704.0   21887   
CHN001             72   3522.0     1789     5458         72  52296.0   20693   
DEL001             72   2568.0     1545     3824         72  52025.0   24155   
HYD001             72   2339.0     1340     3267         72  51109.0   21406   
KOL001             72   3198.0     2097     4517         72  50355.0   27968   
PUN001             72   2511.0     1373     3697         72  51735.0   27711   

           max_in  
branch_id          
AHM001      73945  
CHN001      72109  
DEL001      74465  
HYD001      72092  
KOL001      69168  
PUN001      76477  


In [8]:
cols = ['branch_id', 'warehouse_type', 'warehouse_capacity', 'total_employees', 'monthly_operational_cost']
print(branches[cols])
print()
print(branches[cols].isna().sum())

  branch_id warehouse_type warehouse_capacity  total_employees  \
0    DEL001        Central         45230 sqft               92   
1    PUN001       Regional         38750 sqft               74   
2    CHN001       Regional         41890 sqft               81   
3    HYD001         Branch         36210 sqft               57   
4    KOL001         Branch         29880 sqft               49   
5    AHM001    Service Hub         34120 sqft               63   

   monthly_operational_cost  
0                   3189000  
1                   2654000  
2                   2739000  
3                   1887000  
4                   1743000  
5                   2018000  

branch_id                   0
warehouse_type              0
warehouse_capacity          0
total_employees             0
monthly_operational_cost    0
dtype: int64


In [9]:
per = res[['avg_out', 'avg_in']].join(branches.set_index('branch_id')['total_employees'])
per['out_per_employee'] = (per['avg_out'] / per['total_employees']).round(1)
per['in_per_employee'] = (per['avg_in'] / per['total_employees']).round(1)
print(per)
print()
print('empty values:', per.isna().sum().sum())

           avg_out   avg_in  total_employees  out_per_employee  \
branch_id                                                        
AHM001      2967.0  50704.0               63              47.1   
CHN001      3522.0  52296.0               81              43.5   
DEL001      2568.0  52025.0               92              27.9   
HYD001      2339.0  51109.0               57              41.0   
KOL001      3198.0  50355.0               49              65.3   
PUN001      2511.0  51735.0               74              33.9   

           in_per_employee  
branch_id                   
AHM001               804.8  
CHN001               645.6  
DEL001               565.5  
HYD001               896.6  
KOL001              1027.7  
PUN001               699.1  

empty values: 0


In [10]:
for name, df, dcol in [('sales', sales, 'delivery_date'), ('purch', purch, 'received_date')]:
    print(name, 'rows:', len(df))
    print('  empty in branch_id, product_id, quantity, date:', df[['branch_id', 'product_id', 'quantity', dcol]].isna().sum().sum())
    print('  quantity zero or below:', (df['quantity'] <= 0).sum())
    print('  min and max quantity:', df['quantity'].min(), df['quantity'].max())
    print('  dates that fail to convert:', pd.to_datetime(df[dcol], errors='coerce').isna().sum())

sales rows: 117656
  empty in branch_id, product_id, quantity, date: 0
  quantity zero or below: 0
  min and max quantity: 1 20
  dates that fail to convert: 0
purch rows: 140077
  empty in branch_id, product_id, quantity, date: 0
  quantity zero or below: 0
  min and max quantity: 20 300
  dates that fail to convert: 0


In [11]:
expected = pd.period_range('2019-01', '2024-12', freq='M')
print('months expected:', len(expected))
for name, tbl in [('OUT (mo)', mo), ('IN (mi)', mi)]:
    t = tbl.reindex(expected)
    print(name, '| months in table:', len(tbl), '| branch-months with no units:', t.isna().sum().sum(), '| branch-months at zero:', (t == 0).sum().sum(), '| lowest month:', t.min().min())

months expected: 72
OUT (mo) | months in table: 72 | branch-months with no units: 0 | branch-months at zero: 0 | lowest month: 1294
IN (mi) | months in table: 72 | branch-months with no units: 0 | branch-months at zero: 0 | lowest month: 20693


In [12]:
def dup_test(df, id_col, price_col, label):
    key = [id_col, 'product_id']
    rep = df[df.duplicated(subset=key, keep=False)]
    combos = rep.groupby(key).ngroups
    exact_rows = rep[rep.duplicated(subset=key + ['quantity', price_col], keep=False)]
    exact = exact_rows.groupby(key).ngroups
    print(label, 'rows:', len(df), '| order-product pairs with repeated lines:', combos, '| with an exact copy:', exact, '| share %:', round(100 * exact / combos, 2))

dup_test(sales, 'so_id', 'unit_price', 'sales')
dup_test(purch, 'po_id', 'unit_cost', 'purch')

sales rows: 117656 | order-product pairs with repeated lines: 12184 | with an exact copy: 684 | share %: 5.61
purch rows: 140077 | order-product pairs with repeated lines: 14408 | with an exact copy: 0 | share %: 0.0


In [13]:
key_p = ['po_id', 'product_id']
rp = purch[purch.duplicated(subset=key_p, keep=False)]
pairs_p = rp.groupby(key_p).ngroups
print('purch pairs with repeated lines:', pairs_p)
print('purch pairs with same quantity (cost ignored):', rp[rp.duplicated(subset=key_p + ['quantity'], keep=False)].groupby(key_p).ngroups)
print('purch pairs with same unit_cost (quantity ignored):', rp[rp.duplicated(subset=key_p + ['unit_cost'], keep=False)].groupby(key_p).ngroups)
print('purch distinct unit_cost values per pair (count of pairs):')
print(rp.groupby(key_p)['unit_cost'].nunique().value_counts())

key_s = ['so_id', 'product_id']
rs = sales[sales.duplicated(subset=key_s, keep=False)].copy()
rs['n'] = rs.groupby(key_s)['quantity'].transform('size')
for lab, sub in [('2 lines', rs[rs['n'] == 2]), ('3 or more lines', rs[rs['n'] >= 3])]:
    pairs = sub.groupby(key_s).ngroups
    ex = sub[sub.duplicated(subset=key_s + ['quantity', 'unit_price'], keep=False)].groupby(key_s).ngroups
    print('sales', lab, '| pairs:', pairs, '| exact copy:', ex, '| share %:', round(100 * ex / pairs, 2))

purch pairs with repeated lines: 14408
purch pairs with same quantity (cost ignored): 66
purch pairs with same unit_cost (quantity ignored): 0
purch distinct unit_cost values per pair (count of pairs):
unit_cost
2    13219
3     1129
4       57
5        3
Name: count, dtype: int64
sales 2 lines | pairs: 11202 | exact copy: 547 | share %: 4.88
sales 3 or more lines | pairs: 982 | exact copy: 137 | share %: 13.95


In [14]:
x_s = so_l[so_l['so_id'].isin(so_h.loc[so_h['order_status'] != 'Delivered', 'so_id'])]
x_p = po_l[po_l['po_id'].isin(po_h.loc[po_h['po_status'] != 'Received', 'po_id'])]

print('SALES lines: all', len(so_l), '| counted', len(sales), '| left out', len(x_s), '| difference:', len(so_l) - len(sales) - len(x_s))
print('SALES units: all', so_l['quantity'].sum(), '| counted', sales['quantity'].sum(), '| left out', x_s['quantity'].sum(), '| difference:', so_l['quantity'].sum() - sales['quantity'].sum() - x_s['quantity'].sum())
print('SALES counted units: months 2019-2024', mo.sum().sum(), '+ 2025', sales.loc[sales['year'] == 2025, 'quantity'].sum(), '| difference:', sales['quantity'].sum() - mo.sum().sum() - sales.loc[sales['year'] == 2025, 'quantity'].sum())

print('PURCH lines: all', len(po_l), '| counted', len(purch), '| left out', len(x_p), '| difference:', len(po_l) - len(purch) - len(x_p))
print('PURCH units: all', po_l['quantity'].sum(), '| counted', purch['quantity'].sum(), '| left out', x_p['quantity'].sum(), '| difference:', po_l['quantity'].sum() - purch['quantity'].sum() - x_p['quantity'].sum())
print('PURCH counted units: months 2019-2024', mi.sum().sum(), '+ 2025', purch.loc[purch['year'] == 2025, 'quantity'].sum(), '| difference:', purch['quantity'].sum() - mi.sum().sum() - purch.loc[purch['year'] == 2025, 'quantity'].sum())

SALES lines: all 130402 | counted 117656 | left out 12746 | difference: 0
SALES units: all 1368534 | counted 1234939 | left out 133595 | difference: 0
SALES counted units: months 2019-2024 1231567 + 2025 3372 | difference: 0
PURCH lines: all 155495 | counted 140077 | left out 15418 | difference: 0
PURCH units: all 24867716 | counted 22387123 | left out 2480593 | difference: 0
PURCH counted units: months 2019-2024 22192169 + 2025 194954 | difference: 0


In [15]:
so_branch = dict(zip(so_h['so_id'], so_h['branch_id']))
po_branch = dict(zip(po_h['po_id'], po_h['branch_id']))
so_ok = set(so_h.loc[so_h['order_status'] == 'Delivered', 'so_id'])
po_ok = set(po_h.loc[po_h['po_status'] == 'Received', 'po_id'])
ls = so_l[so_l['so_id'].isin(so_ok)]
lp = po_l[po_l['po_id'].isin(po_ok)]
out2 = ls.groupby(ls['so_id'].map(so_branch))['quantity'].sum()
in2 = lp.groupby(lp['po_id'].map(po_branch))['quantity'].sum()

out1 = sales.groupby('branch_id')['quantity'].sum()
in1 = purch.groupby('branch_id')['quantity'].sum()

print('lines used, second method: OUT', len(ls), ' IN', len(lp))
print('units OUT per branch, second method:'); print(out2)
print('units IN per branch, second method:'); print(in2)
print('units OUT, largest difference between the two methods:', (out1 - out2).abs().max())
print('units IN, largest difference between the two methods:', (in1 - in2).abs().max())

b = list(res.index)
avg_out2 = ((out_y.loc[b, 'TOTAL'] - out_y.loc[b, 2025]) / 72).round(0)
avg_in2 = ((in_y.loc[b, 'TOTAL'] - in_y.loc[b, 2025]) / 72).round(0)
print('monthly average OUT, largest difference:', (res['avg_out'] - avg_out2).abs().max())
print('monthly average IN, largest difference:', (res['avg_in'] - avg_in2).abs().max())

emp = branches.set_index('branch_id')['total_employees']
print('OUT per employee, largest difference:', (per['out_per_employee'] - (avg_out2 / emp).round(1)).abs().max())
print('IN per employee, largest difference:', (per['in_per_employee'] - (avg_in2 / emp).round(1)).abs().max())

lines used, second method: OUT 117656  IN 140077
units OUT per branch, second method:
so_id
AHM001    214223
CHN001    253921
DEL001    185319
HYD001    168975
KOL001    230868
PUN001    181633
Name: quantity, dtype: int64
units IN per branch, second method:
po_id
AHM001    3681090
CHN001    3797868
DEL001    3777619
HYD001    3718428
KOL001    3661187
PUN001    3750931
Name: quantity, dtype: int64
units OUT, largest difference between the two methods: 0
units IN, largest difference between the two methods: 0
monthly average OUT, largest difference: 0.0
monthly average IN, largest difference: 0.0
OUT per employee, largest difference: 0.0
IN per employee, largest difference: 0.0


In [16]:
def check6(df, datecol, name):
    d = df[['branch_id', datecol, 'quantity']].copy()
    d['month'] = pd.to_datetime(d[datecol]).dt.to_period('M')
    d = d[d['month'] <= '2024-12']
    print(name, '- lines in full months (Jan 2019 to Dec 2024):', len(d))
    g = d.groupby('branch_id')['quantity']
    r = pd.DataFrame({'mean_line': g.mean().round(1), 'max_line': g.max()})
    r['max_vs_mean'] = (r['max_line'] / r['mean_line']).round(1)
    print(r)
    m = d.groupby(['branch_id', 'month']).agg(units=('quantity', 'sum'), lines=('quantity', 'size'), biggest=('quantity', 'max'))
    m['biggest_share_%'] = (m['biggest'] / m['units'] * 100).round(2)
    avg_lines = m.groupby(level=0)['lines'].mean().round(0)
    rows = []
    for b, x in m.groupby(level=0):
        for label, idx in [('busiest', x['units'].idxmax()), ('quietest', x['units'].idxmin())]:
            rows.append([b, label, str(idx[1]), x.loc[idx, 'units'], x.loc[idx, 'lines'], avg_lines[b], x.loc[idx, 'biggest'], x.loc[idx, 'biggest_share_%']])
    print(pd.DataFrame(rows, columns=['branch', 'month_type', 'month', 'units', 'lines', 'avg_lines_per_month', 'biggest_line', 'biggest_share_%']))

check6(sales, 'delivery_date', 'SALES (OUT)')
check6(purch, 'received_date', 'PURCHASES (IN)')

SALES (OUT) - lines in full months (Jan 2019 to Dec 2024): 117316
           mean_line  max_line  max_vs_mean
branch_id                                  
AHM001          10.5        20          1.9
CHN001          10.5        20          1.9
DEL001          10.6        20          1.9
HYD001          10.5        20          1.9
KOL001          10.5        20          1.9
PUN001          10.4        20          1.9
    branch month_type    month  units  lines  avg_lines_per_month  \
0   AHM001    busiest  2019-06   4150    381                283.0   
1   AHM001   quietest  2022-11   1294    123                283.0   
2   CHN001    busiest  2019-08   5458    499                336.0   
3   CHN001   quietest  2021-09   1789    168                336.0   
4   DEL001    busiest  2022-01   3824    355                243.0   
5   DEL001   quietest  2019-01   1545    137                243.0   
6   HYD001    busiest  2021-12   3267    310                223.0   
7   HYD001   quietest  2022-07

In [17]:
for name, df, dcol in [('SALES', sales, 'delivery_date'), ('PURCH', purch, 'received_date')]:
    d = pd.to_datetime(df[dcol])
    o = pd.to_datetime(df['order_date'])
    print(name, '- earliest order_date:', o.min().date(), '| latest order_date:', o.max().date())
    print(name, '-', dcol, 'earliest:', d.min().date(), '| latest:', d.max().date())
    n25 = (d.dt.year == 2025).sum()
    print(name, '- lines dated 2025:', n25, '| dates in 2025 run from', d[d.dt.year == 2025].min().date(), 'to', d[d.dt.year == 2025].max().date())
    print(name, '- lines in 2019-01, 2019-02, 2019-03:', [int((d.dt.to_period('M') == p).sum()) for p in ['2019-01', '2019-02', '2019-03']])
    print(name, '- lines in 2024-10, 2024-11, 2024-12:', [int((d.dt.to_period('M') == p).sum()) for p in ['2024-10', '2024-11', '2024-12']])
    print()

SALES - earliest order_date: 2019-01-01 | latest order_date: 2024-12-31
SALES - delivery_date earliest: 2019-01-03 | latest: 2025-01-13
SALES - lines dated 2025: 340 | dates in 2025 run from 2025-01-01 to 2025-01-13
SALES - lines in 2019-01, 2019-02, 2019-03: [1132, 1385, 1715]
SALES - lines in 2024-10, 2024-11, 2024-12: [1790, 1558, 1559]

PURCH - earliest order_date: 2019-01-01 | latest order_date: 2024-12-31
PURCH - received_date earliest: 2019-01-11 | latest: 2025-01-28
PURCH - lines dated 2025: 1246 | dates in 2025 run from 2025-01-01 to 2025-01-28
PURCH - lines in 2019-01, 2019-02, 2019-03: [899, 1831, 2197]
PURCH - lines in 2024-10, 2024-11, 2024-12: [1831, 1954, 2045]



In [18]:
for label, df, dcol in [('units OUT (sales)', sales, 'delivery_date'), ('units IN (purchases)', purch, 'received_date')]:
    d = df[['branch_id', dcol, 'quantity']].copy()
    d['month'] = pd.to_datetime(d[dcol]).dt.to_period('M')
    d = d[d['month'] <= '2024-12']
    m = d.groupby(['branch_id', 'month'])['quantity'].sum().unstack(0)
    all72 = m.mean()
    without_jan = m.drop(pd.Period('2019-01')).mean()
    out = pd.DataFrame({'average, all 72 months': all72.round(0), 'average, without Jan 2019 (71 months)': without_jan.round(0)})
    out['change in average, %'] = ((without_jan / all72 - 1) * 100).round(2)
    print(label)
    print(out)
    print()

units OUT (sales)
           average, all 72 months  average, without Jan 2019 (71 months)  \
branch_id                                                                  
AHM001                     2967.0                                 2985.0   
CHN001                     3522.0                                 3533.0   
DEL001                     2568.0                                 2583.0   
HYD001                     2339.0                                 2351.0   
KOL001                     3198.0                                 3209.0   
PUN001                     2511.0                                 2520.0   

           change in average, %  
branch_id                        
AHM001                     0.58  
CHN001                     0.34  
DEL001                     0.56  
HYD001                     0.50  
KOL001                     0.37  
PUN001                     0.34  

units IN (purchases)
           average, all 72 months  average, without Jan 2019 (71 months)  \
bra

In [19]:
p = purch.copy()
p['year'] = pd.to_datetime(p['received_date']).dt.year
p = p[p['year'] <= 2024]
g = p.groupby(['branch_id', 'year']).agg(units=('quantity', 'sum'), orders=('po_id', 'nunique'))
g['units_per_order'] = g['units'] / g['orders']

s = g.groupby(level=0).agg(orders_lowest=('orders', 'min'), orders_highest=('orders', 'max'),
                           units_per_order_lowest=('units_per_order', 'min'), units_per_order_highest=('units_per_order', 'max'),
                           units_lowest=('units', 'min'), units_highest=('units', 'max'))
s['orders, highest vs lowest year, %'] = ((s['orders_highest'] / s['orders_lowest'] - 1) * 100).round(1)
s['units per order, highest vs lowest year, %'] = ((s['units_per_order_highest'] / s['units_per_order_lowest'] - 1) * 100).round(1)
s['units, highest vs lowest year, %'] = ((s['units_highest'] / s['units_lowest'] - 1) * 100).round(1)
print(s[['orders, highest vs lowest year, %', 'units per order, highest vs lowest year, %', 'units, highest vs lowest year, %']])
print()
print('across the 36 branch-years, units vs orders received:', round(g['units'].corr(g['orders']), 2))
print('across the 36 branch-years, units vs units per order:', round(g['units'].corr(g['units_per_order']), 2))

           orders, highest vs lowest year, %  \
branch_id                                      
AHM001                                  11.2   
CHN001                                  15.1   
DEL001                                  10.3   
HYD001                                  13.3   
KOL001                                   9.0   
PUN001                                  15.0   

           units per order, highest vs lowest year, %  \
branch_id                                               
AHM001                                            4.9   
CHN001                                            4.9   
DEL001                                            3.7   
HYD001                                            6.4   
KOL001                                            6.6   
PUN001                                            7.3   

           units, highest vs lowest year, %  
branch_id                                    
AHM001                                 16.3  
CHN001              

In [20]:
o = pd.to_datetime(purch['order_date'])
r = pd.to_datetime(purch['received_date'])
lead = (r - o).dt.days
print('lead time in days for purchases: smallest', lead.min(), '| average', round(lead.mean(), 1), '| largest', lead.max())
for p in ['2019-01', '2019-02', '2019-03', '2019-04']:
    print(p, '- lines ORDERED in this month:', int((o.dt.to_period('M') == p).sum()), '| lines RECEIVED in this month:', int((r.dt.to_period('M') == p).sum()))

lead time in days for purchases: smallest 7 | average 17.9 | largest 35
2019-01 - lines ORDERED in this month: 2002 | lines RECEIVED in this month: 899
2019-02 - lines ORDERED in this month: 2019 | lines RECEIVED in this month: 1831
2019-03 - lines ORDERED in this month: 2132 | lines RECEIVED in this month: 2197
2019-04 - lines ORDERED in this month: 1961 | lines RECEIVED in this month: 2049


In [21]:
print(list(so_h.columns))
print(list(po_h.columns))
emp = branches.set_index('branch_id')['total_employees']

def hand(side, branch, year, month):
    if side == 'OUT':
        h, l, key, dcol, ok, okval = so_h, so_l, 'so_id', 'delivery_date', 'order_status', 'Delivered'
    else:
        h, l, key, dcol, ok, okval = po_h, po_l, 'po_id', 'received_date', 'po_status', 'Received'
    h = h[(h['branch_id'] == branch) & (h[ok] == okval)].copy()
    h['d'] = pd.to_datetime(h[dcol])
    h = h[h['d'] <= '2024-12-31']
    total = int(l[l[key].isin(h[key])]['quantity'].sum())
    hm = h[(h['d'].dt.year == year) & (h['d'].dt.month == month)]
    lm = l[l[key].isin(hm[key])]
    return len(lm), int(lm['quantity'].sum()), total

items = [(1, 'OUT', 'AHM001', 2019, 6, 381, 4150),
         (2, 'IN', 'CHN001', 2019, 1, 143, 20693),
         (3, 'IN', 'PUN001', 2023, 11, 492, 76477)]

for n, side, b, y, m, exp_lines, exp_units in items:
    lines, units, total = hand(side, b, y, m)
    avg = round(total / 72)
    per_emp = round(avg / emp[b], 1)
    col_avg = 'avg_out' if side == 'OUT' else 'avg_in'
    col_per = 'out_per_employee' if side == 'OUT' else 'in_per_employee'
    print('Item', n, '-', b, f'{y}-{m:02d}', 'units', side)
    print('  lines in that month: by hand', lines, '| reported', exp_lines, '| difference', lines - exp_lines)
    print('  units in that month: by hand', units, '| reported', exp_units, '| difference', units - exp_units)
    print('  monthly average, 72 months: by hand', avg, '| table', res.loc[b, col_avg], '| difference', avg - res.loc[b, col_avg])
    print('  per employee: by hand', per_emp, '| table', per.loc[b, col_per], '| difference', round(per_emp - per.loc[b, col_per], 1))
    print()

['so_id', 'customer_id', 'branch_id', 'order_date', 'delivery_date', 'order_status', 'payment_terms', 'total_order_value', 'total_gst_amount', 'grand_total', 'sales_channel']
['po_id', 'supplier_id', 'branch_id', 'order_date', 'expected_delivery_date', 'received_date', 'po_status', 'total_cost', 'total_gst_amount', 'grand_total']
Item 1 - AHM001 2019-06 units OUT
  lines in that month: by hand 381 | reported 381 | difference 0
  units in that month: by hand 4150 | reported 4150 | difference 0
  monthly average, 72 months: by hand 2967 | table 2967.0 | difference 0.0
  per employee: by hand 47.1 | table 47.1 | difference 0.0

Item 2 - CHN001 2019-01 units IN
  lines in that month: by hand 143 | reported 143 | difference 0
  units in that month: by hand 20693 | reported 20693 | difference 0
  monthly average, 72 months: by hand 52296 | table 52296.0 | difference 0.0
  per employee: by hand 645.6 | table 645.6 | difference 0.0

Item 3 - PUN001 2023-11 units IN
  lines in that month: by ha